In [16]:
from pytential.sympy_pytential import sympy_pytential
import numpy as np
import matplotlib.pyplot as plt
from pytential.reduce.matrix_methods import reduce_qp

Create two ideal mixing functions from a set of properties, and check them.

In [17]:
from sympy import log, symbols
c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd = symbols('c0a, c0b, c0c, c0d, c1d, c0, c1, Va, Vb, Vc, Vd')

In [18]:
T = 1600
RT = 8.134*T
mu0_SiC = -161028
rho_SiC = 3.21 / 40.11 * 1e6
rho_Ar = 101e3 / RT
v_SiC = 1/rho_SiC
v_Ar = 1/rho_Ar

In [19]:
fa_sp = c0a*mu0_SiC 
fb_sp = c0b*mu0_SiC
fc_sp = c0c*mu0_SiC 
fd_sp = c0d*(-891+RT*log(c0d/(c0d+c1d))) +c1d*(-290457+RT*log(c1d/(c0d+c1d)))

In [20]:
fa = sympy_pytential(fa_sp, constraints_sym=[c0a*v_SiC-Va])
fb = sympy_pytential(fb_sp, constraints_sym=[c0b*v_SiC-Vb])
fc = sympy_pytential(fc_sp, constraints_sym=[c0c*v_SiC-Vc])
fd = sympy_pytential(fd_sp, constraints_sym=[c0d*v_Ar+c1d*v_Ar-Vd])

Make a function fa+fb with all the variables, and add constraints that the concentrations must sum to ca and cb

In [21]:
f = fa+fb+fc+fd
print(f)


Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0a', 'c0b', 'c0c', 'c0d', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + c0d*(13014.4*log(c0d/(c0d + c1d)) - 891) + c1d*(13014.4*log(c1d/(c0d + c1d)) - 290457)

Gradient
[0, 0, 0, 0, -161028, -161028, -161028, 13014.4*log(c0d/(c0d + c1d)) - 891.0, 13014.4*log(c1d/(c0d + c1d)) - 290457.0]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 13014.4*c1d/(c0d*(c0d + c1d)), -13014.4/(c0d + c1d)], [0, 0, 0, 0, 0, 0, 0, -13014.4/(c0d + c1d), 13014.4*c0d/(c1d*(c0d + c1d))]]

Constraints
-Va + 1.24953271028037e-5*c0a
-Vb + 1.24953271028037e-5*c0b
-Vc + 1.24953271028037e-5*c0c
-Vd + 0.128855445544554*c0d + 0.128855445544554*c1d



In [23]:
f2 = f.add_constraints_sym([c0a+c0b+c0c+c0d-c0, c1d-c1])
print(f2)


Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + c0d*(13014.4*log(c0d/(c0d + c1d)) - 891) + c1d*(13014.4*log(c1d/(c0d + c1d)) - 290457)

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 13014.4*log(c0d/(c0d + c1d)) - 891.0, 0, 13014.4*log(c1d/(c0d + c1d)) - 290457.0]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 13014.4*c1d/(c0d*(c0d + c1d)), 0, -13014.4/(c0d + c1d)], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -13014.4/(c0d + c1d), 0, 13014.4*c0d/(c1d*(c0d + c1d))]]

Constraints
-Va + 1.24953271028037e-5*c0a
-Vb + 1.24953271028037e-5*c0b
-Vc + 1.24953271028037e-5*c0c
-Vd + 0.128855445544554*c0d + 0.128855445544554*c1d


In [24]:
y0 = {'Va':1, 'Vb':1, 'Vc':1, 'Vd':1, 'c0a':rho_SiC, 'c0b':rho_SiC, 'c0c':rho_SiC , 'c0d':rho_Ar/100, 'c1d':rho_Ar*(1-1/100), 'c0':0,'c1':0}
f2(**y0)

-38663410089.443985

In [25]:
B = np.array(f2.hess(**y0),dtype=np.float64)
b = np.array(f2.grad(**y0),dtype=np.float64)
A = np.array(f2.get_constraint_jacobian(),dtype=np.float64)
print(B)
print(A)
print(f2.vars)

[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00  

In [26]:
fq = f2.quadratic_expansion(y0)
print(fq)


Variables
['Va', 'Vb', 'Vc', 'Vd', 'c0', 'c0a', 'c0b', 'c0c', 'c0d', 'c1', 'c1d']

Potential
-161028*c0a - 161028*c0b - 161028*c0c + 0.5*c0d*(166020.65473901*c0d - 1676.97631049505*c1d) - 60824.5268685234*c0d + 0.5*c1d*(-1676.97631049505*c0d + 16.9391546514651*c1d) - 290587.799090932*c1d - 38663410089.444

Gradient
[0, 0, 0, 0, 0, -161028, -161028, -161028, 166020.65473901*c0d - 1676.97631049505*c1d - 60824.5268685234, 0, -1676.97631049505*c0d + 16.9391546514651*c1d - 290587.799090932]

Hessian
[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 166020.654739010, 0, -1676.97631049505], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, -1676.97631049505, 0, 16.9391546514651]]

Constraints
-Va + 1.24953271028037e-5*c0a
-Vb + 1.

In [27]:
print(y0)

{'Va': 1, 'Vb': 1, 'Vc': 1, 'Vd': 1, 'c0a': 80029.9177262528, 'c0b': 80029.9177262528, 'c0c': 80029.9177262528, 'c0d': 0.07760634374231619, 'c1d': 7.683028030489303, 'c0': 0, 'c1': 0}


In [28]:
fr = fq.remove_linear_constraints(['c0', 'c1', 'Va', 'Vb', 'Vc', 'Vd'], y0=y0)
print(fr)
fr.write_to_file('SiC_EQ')

Q_tilde is symmetric.

Variables
['c0', 'c1', 'Va', 'Vb', 'Vc', 'Vd']

Potential
0.5*Va*(0.0108006610823854*Va + 0.0108006638755134*Vb + 0.0108006620531299*Vc - 328.627158759576*Vd - 1.34957798839498e-7*c0 + 42.7731302639957*c1) - 27.4348125660189*Va + 0.5*Vb*(0.0108006638755134*Va + 0.0108006666686422*Vb + 0.0108006648462582*Vc - 328.627243744904*Vd - 1.34957833740548e-7*c0 + 42.7731413254323*c1) - 27.4348240700251*Vb + 0.5*Vc*(0.0108006620531299*Va + 0.0108006648462582*Vb + 0.0108006630238746*Vc - 328.62718829601*Vd - 1.34957810969269e-7*c0 + 42.7731341083698*c1) - 27.4348135698958*Vc + 0.5*Vd*(-328.627158759576*Va - 328.627243744904*Vb - 328.62718829601*Vc + 9998999.93626497*Vd + 0.00410630401664963*c0 - 1301439.99174592*c1) + 777642.515341124*Vd + 0.5*c0*(-1.34957798839498e-7*Va - 1.34957833740548e-7*Vb - 1.34957810969269e-7*Vc + 0.00410630401664963*Vd + 1.68634191265446e-12*c0 - 0.000534464276387519*c1) - 161027.999657193*c0 + 0.5*c1*(42.7731302639957*Va + 42.7731413254323*Vb + 42

Visualization

In [12]:
from numpy import linspace, vectorize, meshgrid
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

c0 = linspace(0.001, 0.999, 3)
va = linspace(0.001, 0.999, 3)
C0, Va = meshgrid(c0, va)

args_matrix = np.array([C0.flatten(), (1 - C0).flatten(), Va.flatten(), (1 - Va).flatten()]).T
print(args_matrix)
# Compute Z values using the fa function
F_flat = f2(args_matrix)
print(F_flat)
F = F_flat.reshape(C0.shape)

[[0.001 0.999 0.001 0.999]
 [0.5   0.5   0.001 0.999]
 [0.999 0.001 0.001 0.999]
 [0.001 0.999 0.5   0.5  ]
 [0.5   0.5   0.5   0.5  ]
 [0.999 0.001 0.5   0.5  ]
 [0.001 0.999 0.999 0.001]
 [0.5   0.5   0.999 0.001]
 [0.999 0.001 0.999 0.001]]


ValueError: not enough values to unpack (expected 11, got 9)

In [ ]:
import sympy as sp
x = sp.symbols('x')
f = sp.log(x)
print(f)
fcn = sp.lambdify(x, f, 'scipy')
fcn(1e-6)

fr = replace_log_with_log1p(f)
print(fr)
fcnr = sp.lambdify(x, fr, 'scipy')
fcnr(0)

In [ ]:
def replace_log_with_log1p(expr):
    from sympy.codegen.cfunctions import log1p
    return expr.replace(sp.log, lambda arg: log1p(arg - 1))

In [ ]:
fr = replace_log_with_log1p(f)
print(fr)


In [ ]:
fcn = sp.lambdify(x, f, 'scipy')
print(f)
f(.4)

In [ ]:
#fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'surface'}, {'type': 'xy'}]])
fig = go.Figure()

# Add the 3D surface plot for f
fig.add_trace(go.Surface(
    z=F, 
    x=c0, 
    y=va, 
    colorscale='Viridis',
    contours={
        "x": {"show": True, "color": "white"},
        "y": {"show": True, "color": "white"},
        "z": {"show": False, "color": "white"}
    }
))

fig.add_trace(go.Scatter3d(x=c0, y=[1]*len(c0), z=fa(c0a=c0,c1a=1-c0,Va=1), mode='lines', name='fa', line=dict(color='red', width=4)))
fig.add_trace(go.Scatter3d(x=c0, y=[0]*len(c0), z=fb(c0b=c0,c1b=1-c0,Vb=1), mode='lines', name='fb', line=dict(color='blue', width = 4)))

# # Add the 2D plot for f_reduced
# fig.add_trace(go.Scatter(x=c0, y=fa(c0a=c0,c1a=1-c0,Va=1), mode='lines', name='fa'), row=1, col=2)
# fig.add_trace(go.Scatter(x=c0, y=fb(c0b=c0,c1b=1-c0,Vb=1), mode='lines', name='fb'), row=1, col=2)
# fig.add_trace(go.Scatter(x=c0, y=f2(c0=c0,c1=1-c0,Va=1, Vb=0), mode='lines', name='fb'), row=1, col=2)

# Update layout for the entire figure
fig.update_layout(
    title='3D Surface Plot of f and 2D Plot of f_reduced',
    scene=dict(
        xaxis_title='c0',
        yaxis_title='Va',
        zaxis_title='f',
        aspectratio = dict(x=2, y=2, z=1),
        camera=dict(projection=dict(type="orthographic"))
    ),
)
fig.show()

In [ ]:
f3 = min_pytential(f, vars_out=['Va', 'Vb'])

In [ ]:

f3(Va=.5,Vb=.5)